# All-atom polyethylene melt: MuPT → flowerMD AA-DPD → MuPT → MD input files

This notebook builds a small polyethylene (PE) melt end to end:

| Step | Tool | What happens |
|---|---|---|
| 1–3 | MuPT | build repeat units from SMILES and link them into chains |
| 4 | MuPT → mBuild | export the atoms (with their MuPT addresses) |
| 5 | flowerMD | pack the chains at melt density and relax them with AA-DPD |
| 6 | mBuild → MuPT | write the relaxed coordinates back onto the MuPT representation |
| 7 | MuPT → MDAnalysis | write a PDB with chains and residues |
| 8 | OpenFF Interchange | write GROMACS, LAMMPS and OpenMM input files |

**Environment:** `conda-envs/workflows.yml` (or `conda-envs/mupt-examples.yml`). Runs in about a minute on a laptop CPU.

> This uses the refactored MuPT API that is still under review (MuPT-hub/mupt#141 through #166), so names may still change before release.

In [ ]:
from networkx import connected_components, get_node_attributes, spring_layout
from networkx.drawing import draw as nx_draw

from mupt.graphs.generators import path_graph_from_sequence
from mupt.interfaces.smiles import primitive_from_smiles
from mupt.mupr.connection.alignment import ConnectorAntialignmentRigid
from mupt.mupr.primitives import CompositePrimitive, RootPrimitive
from mupt.mupr.primitives.predicates import is_atom
from mupt.mupr.primitives.visualize import visualize_cross_section

## 1. System size

In [ ]:
num_chains: int = 2
DOP: int = 10  # repeat units per chain

## 2. Repeat units from SMILES

Each repeat unit is a SMILES fragment. `[*:1]` and `[*:2]` mark where it bonds to its neighbors (MuPT turns them into **Connectors**); the end groups cap one side with hydrogen instead.
`embed_positions=True` gives every atom 3D coordinates, which flowerMD needs to start from.

In [ ]:
lexicon = {
    "head": primitive_from_smiles("[H:1]-[C:1](-[H])(-[H])-[C:2](-[H])(-[H])-[*:2]", label="head", embed_positions=True),
    "mid":  primitive_from_smiles("[*:1]-[C:1](-[H])(-[H])-[C:2](-[H])(-[H])-[*:2]", label="mid", embed_positions=True),
    "tail": primitive_from_smiles("[*:1]-[C:1](-[H])(-[H])-[C:2](-[H])(-[H])-[H:2]", label="tail", embed_positions=True),
}
sequence = ["head"] + (DOP - 2) * ["mid"] + ["tail"]

sequence_graph = path_graph_from_sequence(sequence, label_attr="label")
nx_draw(sequence_graph, with_labels=True, labels=get_node_attributes(sequence_graph, "label"))

## 3. Link repeat units into chains

Each copied repeat unit is placed on its neighbor's free Connector (`ConnectorAntialignmentRigid`), then bonded with `connect_neighbor`.
The chain is tagged `role="segment"` and each repeat unit `role="residue"`; exporters use these roles to name chains and residues.

> `populate_from_topology_and_lexicon` will do this in one call once it can link repeat units that already have coordinates; until then this short loop does the same job.

In [ ]:
def build_chain(sequence, bond_length=1.54):
    # Place each embedded repeat unit on its neighbor's Connector, then bond them.
    chain = CompositePrimitive(label="PE", role="segment")
    units = [lexicon[name].copy() for name in sequence]
    for unit in units:
        unit.role = "residue"
        chain.attach_child(unit)

    align = ConnectorAntialignmentRigid()
    for a, b in zip(units, units[1:]):
        x, y = next(
            (x, y)
            for x in a.connections.connectors_free
            for y in b.connections.connectors_free
            if x.bondable_with(y)
        )
        x.set_bond_length(bond_length)
        y.set_bond_length(bond_length)
        b.rigidly_transform(align.antialignment_transformation(align_connector=y, to_connector=x))
        a.connect_neighbor(b, our_connector=x, their_connector=y)
    return chain


chain_prototype = build_chain(sequence)
system = RootPrimitive(label="PE melt", role="universe")
for _ in range(num_chains):
    system.attach_child(chain_prototype.copy())

print(chain_prototype.hierarchy_summary())

## 4. Inspect the atom-level bond graph, then export it to mBuild

`cross_section(is_atom)` slices the hierarchy at the atom level and returns its bond graph: one connected component per chain.

In [ ]:
xsec = system.cross_section(is_atom)
print(f"{len(list(connected_components(xsec)))} chains, {xsec.number_of_nodes()} atoms, {xsec.number_of_edges()} bonds")
visualize_cross_section(system, xsec, pos=spring_layout(xsec, dim=2, seed=1), with_labels=False, node_size=30)

In [ ]:
from mupt.interfaces.mosdef.mbuild import primitive_to_mbuild

compound = primitive_to_mbuild(
    system,
    predicate=is_atom,
    preserve_intermediates=True,  # keep chain -> residue -> atom nesting in the Compound
    coords_to_nm=0.1,  # MuPT coordinates are in Angstrom; mBuild/flowerMD use nm
)
print(compound)

## 5. Pack and relax the melt with flowerMD AA-DPD

flowerMD gets one `Molecule` per MuPT chain. `AtomisticRandomWalk` packs them at the target density with Boltzmann-sampled torsions, `UFF_DPD` sets soft DPD forces, and `PhantomWalk` runs DPD until atoms no longer overlap (and checks that no chain pierces a ring).

In [ ]:
import warnings

import numpy as np
import unyt as u
from flowermd.base import Molecule
from flowermd.library import UFF_DPD, AtomisticRandomWalk, LocalBoltzmannSampler, PhantomWalk

warnings.filterwarnings("ignore")

chains = [  # one flowerMD Molecule per MuPT chain
    Molecule(num_mols=1, compound=chain, name="PE", identify_connections=True)
    for chain in compound.children
]
walk = AtomisticRandomWalk(
    chains,
    density=0.78 * u.g / u.cm**3,
    torsion_sampler=LocalBoltzmannSampler(temperature=413),
    seed=1,
)
print("atoms:", walk.n_particles, " box (nm):", walk.box.Lx)

ff = UFF_DPD(walk, file_path="pe_dpd.xml")
walk.apply_forcefield(r_cut=0.35, force_field=ff, kT=4.184)

In [ ]:
sim = PhantomWalk(
    walk.hoomd_snapshot,
    walk.hoomd_forcefield,
    n_steps_dpd=40000,
    n_steps_fire=100,
    dt=2e-4,
    gsd_write_freq=int(1e5),
    log_write_freq=int(1e5),
    run_now=False,
)
record = sim.run_initialization(record_file="pe_record.json")
print("DPD converged:", record["dpd_converged"], "after", record["dpd_steps"], "steps")
print("ring piercings:", record["ring_piercings"])

## 6. Put the relaxed coordinates back on the MuPT representation

flowerMD keeps the particle order of the exported Compound, so we copy the final HOOMD positions onto that Compound and let `update_primitive_from_mbuild` write them onto the matching Primitives by address.

HOOMD wraps atoms into the periodic box, which splits chains across its faces; adding back each atom's box `image` unwraps them first.

In [ ]:
from mupt.interfaces.mosdef.mbuild import update_primitive_from_mbuild

snapshot = sim.state.get_snapshot()
box_nm = np.asarray(snapshot.configuration.box[:3])
unwrapped_nm = snapshot.particles.position + snapshot.particles.image * box_nm

compound.xyz = unwrapped_nm
moved = update_primitive_from_mbuild(system, compound, coords_to_nm=0.1)
print(f"updated {len(moved)} atoms on the MuPT representation")

## 7. Write a PDB from MuPT

`primitive_to_mdanalysis` uses the roles from step 3: each `segment` becomes a chain and each `residue` a residue, named here with three-letter codes.

In [ ]:
from mupt.interfaces.mdanalysis import primitive_to_mdanalysis

universe = primitive_to_mdanalysis(system, {"head": "PEH", "mid": "PEM", "tail": "PET"})
universe.dimensions = [*(box_nm * 10), 90.0, 90.0, 90.0]  # Angstrom
universe.atoms.write("pe_melt.pdb")

bond_lengths = np.linalg.norm(
    universe.atoms.positions[universe.bonds.indices[:, 0]] - universe.atoms.positions[universe.bonds.indices[:, 1]],
    axis=1,
)
print(f"{universe.atoms.n_atoms} atoms, {universe.residues.n_residues} residues, {universe.segments.n_segments} chains")
print(f"bond lengths {bond_lengths.min():.2f}-{bond_lengths.max():.2f} A")

## 8. MD input files with OpenFF Interchange

OpenFF assigns force field parameters (here OpenFF Sage 2.3.0) and writes inputs for several engines.
Each MuPT chain reaches OpenFF as an RDKit molecule (via mBuild, until MuPT's own RDKit exporter is ported to the refactored API); positions come from the MuPT representation.

> Gasteiger charges keep this example fast. Use AM1-BCC (OpenFF's default) or NAGL charges for production.

In [ ]:
from openff.interchange import Interchange
from openff.toolkit import ForceField, Molecule as OFFMolecule, Topology
from openff.units import unit
from rdkit import Chem

off_molecules = []
for chain in system.children:
    rdmol = primitive_to_mbuild(chain, predicate=is_atom, coords_to_nm=0.1).to_rdkit()
    Chem.SanitizeMol(rdmol)
    off_molecules.append(OFFMolecule.from_rdkit(rdmol, allow_undefined_stereo=True, hydrogens_are_explicit=True))

charged = off_molecules[0]  # every chain is the same molecule
charged.assign_partial_charges("gasteiger")

topology = Topology.from_molecules(off_molecules)
topology.box_vectors = box_nm * unit.nanometer
interchange = Interchange.from_smirnoff(ForceField("openff-2.3.0.offxml"), topology, charge_from_molecules=[charged])
interchange.positions = universe.atoms.positions * unit.angstrom  # same atom order as the MuPT export

interchange.to_gromacs("pe_melt")  # pe_melt.gro / pe_melt.top
interchange.to_lammps("pe_melt")  # pe_melt.lmp / pe_melt_pointenergy.in
openmm_system = interchange.to_openmm_system()
print("OpenMM system with", openmm_system.getNumParticles(), "particles")